# Begin

In [1]:
import os 
import sys 
import copy
from collections import namedtuple, defaultdict, Counter 
import itertools
import json
import datetime
import pprint
import re
import pickle
import dataclasses 
from dataclasses import dataclass 
import IPython
from enum import Flag, StrEnum, auto 

from tqdm.notebook import tqdm
import numpy as np
import einops
import pandas as pd

import torch

project_root_path = '${PROJECT_ROOT_PATH}' 
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append('.')
from bundle import *
sys.path.append(os.path.join(project_root_path, 'lib'))
from utils import * 
from logging_utils import *
from model_registry import *

# Setup

In [2]:
LOG = Logging.get()
RNG = np.random.default_rng()

class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

CONFIG = namedtuple('Config', 
                    'project_root_path, project_root_uri, subproject_path, data_path, private_data_path, run_path, ' + 
                    'self_fname, self_name, ' +
                    'subproject_name,' +
                    'is_cuda, cuda_device, exec_mode, is_interactive')(
    project_root_path=project_root_path,
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    subproject_path=os.path.abspath('.'),
    data_path=os.path.join(project_root_path, 'data'),
    private_data_path=None,
    run_path='',
    self_fname='',
    self_name='',
    subproject_name='',
    is_cuda=torch.cuda.is_available(),
    cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
    exec_mode=ExecMode.MASTER_NOTEBOOK,
    is_interactive=True,
)

if IPython.get_ipython() is None:
    module_fname = __file__
    module_basename = os.path.basename(module_fname)
    module_name, _ = os.path.splitext(module_basename)
    
    CONFIG = CONFIG._replace(self_fname=module_fname, self_name=module_name)
    CONFIG = CONFIG._replace(exec_mode=ExecMode.LAUNCH_MODULE)
else:
    with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
        notebook_fname = json.load(cf)['jupyter_session']
        notebook_basename = os.path.basename(notebook_fname)
        notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    
        m = re.match(r'(\w+)-Copy\d+$', notebook_name)
    
        if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook

        CONFIG = CONFIG._replace(self_fname=notebook_fname, self_name=notebook_name)
        
        is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
        CONFIG = CONFIG._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)

CONFIG = CONFIG._replace(is_interactive=CONFIG.exec_mode != ExecMode.LAUNCH_MODULE)

LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)

CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.self_fname)))
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG = CONFIG._replace(private_data_path=os.path.join(CONFIG.data_path, CONFIG.subproject_name))
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)

os.makedirs(CONFIG.private_data_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurovision',
 'project_root_uri': 'com.develorium.neurovision',
 'subproject_path': '/home/misha/dev/mine/neurovision/12_rnn',
 'data_path': '/home/misha/dev/mine/neurovision/data',
 'private_data_path': '/home/misha/dev/mine/neurovision/data/12_rnn',
 'run_path': '/home/misha/dev/mine/neurovision/run/12_rnn',
 'self_fname': '/home/misha/dev/mine/neurovision/12_rnn/make_bundle.ipynb',
 'self_name': 'make_bundle',
 'subproject_name': '12_rnn',
 'is_cuda': False,
 'cuda_device': 'cpu',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Types

In [3]:
class TokenLevel(StrEnum):
    SYMBOL = auto()
    WORD = auto()

# TextPreprocessor

In [44]:
class TextPreprocessor:
    def __init__(self, token_level):
        match token_level:
            case TokenLevel.SYMBOL:
                self.split_line = self.split_line_symbol
                self.check_token = self.check_token_symbol
            case TokenLevel.WORD:
                self.split_line = self.split_line_word
                self.check_token = self.check_token_word
            case _:
                assert False, f'Unsupported {token_level=}'
        
    def remove_pg_envelope(self, text):
        start_indx = text.find('START OF THIS PROJECT')
        start_indx = text.find('START OF THE PROJECT') if start_indx < 0 else start_indx
        end_indx = text.find('End of the Project Gutenberg')
        end_indx = text.find('END OF THE PROJECT GUTENBERG') if end_indx < 0 else end_indx
        assert start_indx >= 0
        assert end_indx >= 0
        
        while text[start_indx] != '\n': 
            start_indx += 1
        
        while text[start_indx] == '\n':
            start_indx += 1
        
        while text[end_indx] != '\n':
            end_indx -= 1

        return text[start_indx:end_indx]

    SPLIT_CHARS = r'"\',.:;?!()\[\]=\+\*/—%…\-\$£_&'
    SYMS_SUBST_TAB = {
        'ï': 'i', 'é': 'e', 'ô': 'o', 'î': 'i', 'ê': 'e', 'æ': 'e', 'ä': 'a', 'ç': 'c', 'ö': 'o', 'à': 'a', 'ü': 'u', 'è': 'e',
        'œ': 'e', 'ù': 'u', 'ò': 'o', 'ā': 'a', 'á': 'a', 'ñ': 'n', 'ó': 'o', 'ú': 'u', 'û': 'u', '✠': '', '†': 'd.', 'ë': 'e', 'ſ': 's', '•': '-', '°': '', chr(0x338): '',
        
    }
    
    def preprocess(self, text):
        text = re.sub(r'[“”]', '"', text)
        text = re.sub(r'[‘’]', "'", text)
        text = re.sub(r'\r\n', '\n', text)
        text = re.sub(r'([^\n])\n([^\n])', r'\1 \2', text)
        text = re.sub(r'\n(\n)+', '\n', text)
        text = re.sub(r'-(-)+', '. ', text)
        text = re.sub(r'\.\.\.', '…', text)
        text = text.lower()
        text = ''.join(map(lambda sym: TextPreprocessor.SYMS_SUBST_TAB.get(sym, sym), text))
        uniq_syms = Counter(text)
        unsupported_syms = list(filter(lambda sym: not re.match(r'[a-z0-9\s' + TextPreprocessor.SPLIT_CHARS + ']', sym), uniq_syms.keys()))
        assert not unsupported_syms, f'Unsupported syms={unsupported_syms}'
        return text

    
    def split_line_symbol(self, line):
        line = re.sub(r'\s(\s)+', ' ', line)
        return list(filter(self.check_token_symbol, line))
        
    def split_line_word(self, line):
        r = []

        # Combination of line.split() and re.split() is used to retain SPLIT_CHARS as individual tokens but to remove separators (\s)
        # Note: re.split with grouping is a must - in opposite case separators will get lost
        for x in line.split():
            r.extend(filter(None, re.split(r'([' + TextPreprocessor.SPLIT_CHARS + r'])', x))) # filter is used to remove empty elements

        return r

    def check_token_word(self, token):
        return re.match(r'[\w' + TextPreprocessor.SPLIT_CHARS +  r']+', token)

    def check_token_symbol(self, token):
        return re.match(r'[\w ' + TextPreprocessor.SPLIT_CHARS +  r']+', token)

In [45]:
x = "the dog's   bark ran towards him, stopped."
print('|' + '|'.join(TextPreprocessor(TokenLevel.WORD).split_line(x)) + '|')
print('|' + '|'.join(TextPreprocessor(TokenLevel.SYMBOL).split_line(x)) + '|')

|the|dog|'|s|bark|ran|towards|him|,|stopped|.|
|t|h|e| |d|o|g|'|s| |b|a|r|k| |r|a|n| |t|o|w|a|r|d|s| |h|i|m|,| |s|t|o|p|p|e|d|.|


# Generate

## Configure

In [52]:
CHUNK_SIZE = 40
TOKEN_LEVEL = TokenLevel.SYMBOL
BUNDLE_FNAME = os.path.join(CONFIG.private_data_path, f'bundle_{CHUNK_SIZE}_{TOKEN_LEVEL.value}.pkl')
SOURCES = [
    'crime_and_punishment',
    'the_great_gatsby',
    'the_mysterious_island',
    'the_picture_of_dorian_gray',
    'ulysses',
]

## Generate

In [53]:
preprocessed_texts_dir_name = os.path.join(CONFIG.private_data_path, 'preprocessed')
os.makedirs(preprocessed_texts_dir_name, exist_ok=True)

source_to_text = {}
source_to_tokens = {}
df_sources = pd.DataFrame(dict(
    name=SOURCES,
    fname=list(map(lambda s: os.path.join(preprocessed_texts_dir_name, s + '.txt'), SOURCES)),
))

text_preprocessor = TextPreprocessor(TOKEN_LEVEL)

for row in df_sources.itertuples():
    mr = ModelRegistry(maven_group_id='org.gutenberg')
    text = mr.get_asset_content(row.name, 1, 'txt').decode('utf-8')
    text = text_preprocessor.remove_pg_envelope(text)
    text = text_preprocessor.preprocess(text)

    with open(row.fname, 'w', encoding="utf-8") as f:
        f.write(text)

    source_to_text[row.name] = text

vocab = Counter()
chunks = []

for iter_no, iter_name in enumerate(('Vocab', 'Tokens', 'Chunks')):
    if iter_no in [0, 1]:
        for source_name, text in source_to_text.items():
            lines = text.splitlines()
            tokens = [] # for each line will contain list of tokens, i.e. list of lists
            
            for line_ind, line in tqdm(enumerate(lines), total=len(lines), desc=f'{iter_name}, {source_name}'):
                line_tokens = text_preprocessor.split_line(line)
    
                if iter_no == 0:
                    vocab.update(line_tokens)
                else:
                    tokens.append(list(map(lambda t: vocab[t][0], line_tokens)))
                    
            source_to_tokens[source_name] = tokens

        if iter_no == 0:
            text_list, freq_list = zip(*vocab.most_common())
            ind_list = list(range(len(text_list)))
            vocab = dict(zip(text_list, zip(ind_list, freq_list))) # make dict: text -> (ind, freq)
    
    elif iter_no == 2:
        for source_name, tokens in source_to_tokens.items():
            source_ind = df_sources[df_sources.name == source_name].index.item()
            line_inds = [] # aligned with tokens (1-to-1 mapping), stamps every token with a line index

            for line_ind, line_tokens in enumerate(tokens):
                line_inds.extend([line_ind] * len(line_tokens))

            tokens = list(itertools.chain.from_iterable(tokens)) # flatten tokens list
            assert len(line_inds) == len(tokens)
            
            for start_ind in tqdm(range(len(tokens) - CHUNK_SIZE), desc=f'{iter_name}, {source_name}'):
                chunk = tokens[start_ind:start_ind+CHUNK_SIZE]
                assert len(chunk) == CHUNK_SIZE
                chunks.append((np.array(chunk), source_ind, line_inds[start_ind], line_inds[start_ind+CHUNK_SIZE-1]))
    
    else:
        assert False, (iter_no, iter_name)

text_list, ind_freq_list = zip(*vocab.items())
ind_list, freq_list = zip(*ind_freq_list)
df_vocab = pd.DataFrame(dict(
    text=text_list,
    freq=freq_list,
), index=ind_list)
chunk_list, source_id_list, start_line_ind_list, end_line_ind_list = zip(*chunks)
df_chunks = pd.DataFrame(dict(
    chunk=chunk_list,
    source_ind=source_id_list,
    start_line_ind=start_line_ind_list,
    end_line_ind=end_line_ind_list,
))
bundle = Bundle(df_sources=df_sources, df_vocab=df_vocab, df_chunks=df_chunks)

with open(BUNDLE_FNAME, 'wb') as f:
    pickle.dump(bundle, f)
    
LOG(f'Bundle generated and saved to "{BUNDLE_FNAME}"')

Vocab, crime_and_punishment:   0%|          | 0/3970 [00:00<?, ?it/s]

Vocab, the_great_gatsby:   0%|          | 0/1653 [00:00<?, ?it/s]

Vocab, the_mysterious_island:   0%|          | 0/4905 [00:00<?, ?it/s]

Vocab, the_picture_of_dorian_gray:   0%|          | 0/1527 [00:00<?, ?it/s]

Vocab, ulysses:   0%|          | 0/7180 [00:00<?, ?it/s]

...

Chunks, the_great_gatsby:   0%|          | 0/264811 [00:00<?, ?it/s]

Chunks, the_mysterious_island:   0%|          | 0/1102106 [00:00<?, ?it/s]

Chunks, the_picture_of_dorian_gray:   0%|          | 0/426000 [00:00<?, ?it/s]

Chunks, ulysses:   0%|          | 0/1501519 [00:00<?, ?it/s]

Bundle generated and saved to "/home/misha/dev/mine/neurovision/data/12_rnn/bundle_40_symbol.pkl"


# Verify

## Load

In [54]:
%%time

with open(BUNDLE_FNAME, 'rb') as b:
    bundle = pickle.load(b)

chunks = np.array(bundle.df_chunks['chunk'].values.tolist())
assert chunks.shape[1] == CHUNK_SIZE
LOG(f'Bundle with chunks loaded from "{BUNDLE_FNAME}"')

Bundle with chunks loaded from "/home/misha/dev/mine/neurovision/data/12_rnn/bundle_40_symbol.pkl"
CPU times: user 5.98 s, sys: 1.33 s, total: 7.3 s
Wall time: 7.29 s


## Verify vocab

In [55]:
bundle.df_vocab.head()

,text,freq
0,,771179
1,e,425808
2,t,314623
3,a,278218
4,o,268271


In [56]:
bundle.df_vocab.tail()

,text,freq
56,%,6
57,&,4
58,$,2
59,=,2
60,+,1


## Verify chunks

In [57]:
# @launchit.disable
columns = defaultdict(list)
chunks_to_show = 10
chunk_inds = RNG.choice(len(bundle.df_chunks), 10, replace=False)
df_inspect = bundle.df_chunks.loc[chunk_inds]
df_inspect = pd.merge(df_inspect, bundle.df_sources, left_on='source_ind', right_index=True, how='left')
df_inspect.rename(columns={'name': 'source_name', 'fname': 'source_fname'}, inplace=True)
df_inspect['chunk1'] = df_inspect['chunk'].map(lambda c: list(map(lambda t: bundle.df_vocab.iloc[t].text, c)))

sources = {}

for row in df_inspect.itertuples():
    print(f'Chunk #{row.Index}, source={row.source_name} (#{row.source_ind}), lines={row.start_line_ind}:{row.end_line_ind}')
    print('- as int list: |' + '|'.join(map(str, row.chunk)) + '|')
    print('- as str list: |' + '|'.join(map(str, row.chunk1)) + '|')
    
    if not row.source_name in sources:
        with open(row.source_fname, 'r', encoding='utf-8') as f:
            sources[row.source_name] = f.readlines()
        
    print('- origin:      ' + ''.join(sources[row.source_name][row.start_line_ind:row.end_line_ind+1]))
    print()

Chunk #2053781, source=the_mysterious_island (#2), lines=2867:2867
- as int list: |2|7|3|6|0|3|0|24|6|5|16|1|20|0|9|12|8|7|1|10|0|4|6|0|2|7|1|0|16|4|9|13|5|10|3|21|11|1|0|3|
- as str list: |t|h|a|n| |a| |k|n|i|f|e|,| |r|u|s|h|e|d| |o|n| |t|h|e| |f|o|r|m|i|d|a|b|l|e| |a|
- origin:      but the stranger, with no other weapon than a knife, rushed on the formidable animal, who turned to meet this new adversary.


Chunk #383180, source=crime_and_punishment (#0), lines=1270:1270
- as int list: |21|1|17|0|18|4|12|0|2|4|0|14|11|4|8|1|0|2|7|3|2|0|10|4|4|9|0|3|2|0|4|6|14|1|0|3|6|10|0|2|
- as str list: |b|e|g| |y|o|u| |t|o| |c|l|o|s|e| |t|h|a|t| |d|o|o|r| |a|t| |o|n|c|e| |a|n|d| |t|
- origin:      "you are not amalia ivanovna, but amalia ludwigovna, and as i am not one of your despicable flatterers like mr. lebeziatnikov, who's laughing behind the door at this moment (a laugh and a cry of 'they are at it again' was in fact audible at the door) so i shall always call you amalia ludwigovna, though 